# ENTERPRISE KNOWLEDGE MINING SYSTEM

#### CONFIGURATIONS

In [47]:
from dataclasses import dataclass
from pathlib import Path

BASE_URL = "https://export.arxiv.org/api/query"

@dataclass
class PipelineConfig:
    search_query: str = "all:computer"
    start: int = 0
    max_results: int = 2
    pdf_dir: Path = Path("pdfs")
    min_page_length: int = 50
    chunk_size: int = 500
    chunk_overlap: int = 200
    batch_size: int = 20
    embedding_model: str = "text-embedding-3-small"
    rag_model: str = "gpt-4.1-mini"
    chroma_path: str = "./chroma_db"
    collection_name: str = "research_papers"

config = PipelineConfig()
config.pdf_dir.mkdir(exist_ok=True)

params = {
    "search_query": config.search_query,
    "start": config.start,
    "max_results": config.max_results,
}

#### DOCUMENT INGESTION

In [48]:
import xml.etree.ElementTree as ET
import time
import pymupdf4llm
import os
import re

NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom"    # extension elements that are not mapped to standard Atom specification
}


In [49]:
import urllib.request
import urllib.error

# ==== Making Request ====
def make_request(url, delay=5, retries=3):
    for attempt in range(retries):
        try:
            time.sleep(delay)
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "arxiv-data-project/1.0 (research script)"}
            )
            with urllib.request.urlopen(request) as response:
                return response.read()

        except urllib.error.HTTPError as e:
            if e.code == 429:
                wait = delay * (2 ** attempt)
                time.sleep(wait)
            else:
                raise

    raise Exception("Max retries exceeded")

In [50]:
# ==== Parsing Helper Functions ====
def get_text(entry, tag):
    return entry.findtext(tag, default="", namespaces=NS).strip()


def extract_authors(entry):
    return [
        author.find("atom:name", NS).text.strip()
        for author in entry.findall("atom:author", NS)
    ]


def extract_categories(entry):
    return [
        cat.attrib.get("term")
        for cat in entry.findall("atom:category", NS)
        if cat.attrib.get("term")
    ]


def extract_pdf_link(entry):
    return next(
        (
            link.attrib.get("href")
            for link in entry.findall("atom:link", NS)
            if link.attrib.get("title") == "pdf"
            and link.attrib.get("type") == "application/pdf"
            and link.attrib.get("rel") == "related"
        ),
        None
    )


def extract_primary_category(entry):
    primary = entry.find("arxiv:primary_category", NS)
    return primary.attrib.get("term") if primary is not None else None


def extract_paper(entry):
    return {
        "arxiv_id": get_text(entry, "atom:id").split("/")[-1],
        "title": get_text(entry, "atom:title"),
        "summary": get_text(entry, "atom:summary"),
        "published": get_text(entry, "atom:published"),
        "authors": extract_authors(entry),
        "primary_category": extract_primary_category(entry),
        "categories": extract_categories(entry),
        "pdf_link": extract_pdf_link(entry),
    }

In [51]:
def parse_response(response):
    root = ET.fromstring(response)
    return [extract_paper(entry) for entry in root.findall("atom:entry", NS)]

#### EXTRACTING PDF AND CONVERTING TO MARKDOWN

In [52]:
# pdf_dir = "pdfs"
# os.makedirs(pdf_dir, exist_ok = True)
pdf_dir = str(config.pdf_dir)
os.makedirs(pdf_dir, exist_ok=True)


def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", name)


def download_pdf(url, save_path, overwrite=False):
    if os.path.exists(save_path) and not overwrite:
        return save_path

    urllib.request.urlretrieve(url, save_path)
    return save_path


def process_paper(paper):
    if not paper.get("pdf_link"):
        print(f"Skipping {paper['title']} (no PDF)")
        return None
    
    safe_title = sanitize_filename(paper["title"])
    local_pdf = f"{safe_title}.pdf"
    path = os.path.join(pdf_dir, local_pdf)

    pdf_file = download_pdf(paper["pdf_link"], path)
    pages = pymupdf4llm.to_markdown(pdf_file, page_chunks=True)

    return {
        **paper,
        "pages": pages
    }


#### DATA CLEANING FOR CHUNKING

In [53]:
# cleaning functions for headings
def clean_heading(text: str) -> str:
    text = text.strip()
    text = re.sub(r'[*_`#]+', '', text)   # remove markdown markers
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [54]:
# splitting pages into section blocks based on headings and common section titles
def extract_section_blocks(page_text: str):
    pattern = re.compile(
        r'^(##.*|_?\*{0,2}Abstract\*{0,2}_?.*|_?\*{0,2}Keywords:.*\*{0,2}_?)$',
        re.IGNORECASE | re.MULTILINE
    )

    matches = list(pattern.finditer(page_text))
    section_blocks = []

    for i, match in enumerate(matches):
        section = re.sub(r'[*_`#]+', '', match.group()).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(page_text)
        text = page_text[start:end].strip()

        if text:
            section_blocks.append({
                "section": section,
                "text": text
            })

    return section_blocks

In [55]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove markdown heading markers and separator-only lines
    text = re.sub(r"(?m)^\s*#{1,6}\s*", " ", text)
    text = re.sub(r"(?m)^\s*[-*_~=`]{2,}\s*$", " ", text)

    # Remove markdown emphasis symbols anywhere in text
    text = re.sub(r"[_*`~]+", " ", text)

    # Remove hash symbols that can confuse NER
    text = re.sub(r"(?<!\w)#(?=\w)", "", text)
    text = text.replace("#", " ")

    # Fix hyphenated line breaks: learn-\ning -> learning
    text = re.sub(r"([A-Za-z])-\s*\n\s*([A-Za-z])", r"\1\2", text)

    # Remove obvious page-only lines
    text = re.sub(r"(?im)^\s*page\s*\d+(\s*of\s*\d+)?\s*$", " ", text)
    text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)

    # Normalize long dashes
    text = re.sub(r"[—–]+", " - ", text)

    # Remove leading punctuation/whitespace 
    text = re.sub(r"^[\s\.,;:-]+", "", text)

    # Collapse repeated whitespace/newlines
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.replace("\n", " ")

    # Final whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [56]:
def clean_page_text(text: str, page_number=None) -> str:
    text = clean_text(text)

    if not text:
        return ""

    # First-page specific cleanup:
    # if Abstract exists, keep text starting from Abstract
    if page_number == 1:
        match = re.search(r"\babstract\b\s*[:-]?\s*", text, flags=re.IGNORECASE)
        if match:
            text = text[match.start():]

    return re.sub(r"\s+", " ", text).strip()

In [57]:
def clean_processed_papers(processed_papers, min_length):
    cleaned_papers = []
    
    for paper in processed_papers:
        clean_pages = []

        for page in paper["pages"]:
            raw_text = page.get("text", "")
            cleaned = clean_page_text(
                raw_text,
                page_number=page["metadata"].get("page_number")
            )

            if cleaned and len(cleaned) >= min_length:  # filter out very short/empty pages
                clean_pages.append({
                    "text": cleaned,
                    "raw_text": raw_text,
                    "metadata": page["metadata"]
                })

        cleaned_papers.append({
            **{key: value for key, value in paper.items() if key != "pages"},
            "pages": clean_pages
        })
        
    return cleaned_papers


In [58]:
import urllib.parse
from debug_utils import preview_cleaned_paper, preview_section_blocks


def process_pdf_to_chunks(params, base_url=BASE_URL, min_length=50, max_papers=None):
    query_url = base_url + "?" + urllib.parse.urlencode(params)

    response = make_request(query_url)
    papers = parse_response(response)

    if max_papers is not None:
        papers = papers[:max_papers]

    processed_papers = []
    for paper in papers:
        result = process_paper(paper)
        if result:
            processed_papers.append(result)

    cleaned_papers = clean_processed_papers(processed_papers, min_length)

    return cleaned_papers   


# cleaned_papers = process_pdf_to_chunks(params)
cleaned_papers = process_pdf_to_chunks(
    params=params,
    min_length=config.min_page_length,
    max_papers=config.max_results
)
preview_cleaned_paper(cleaned_papers)


OCR disabled because Tesseract language data not found.
OCR disabled because Tesseract language data not found.

==== Vision Based Game Development Using Human Computer Interaction ====

--- Page 0 Preview ---
Metadata: {'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using Human Computer Interaction.pdf', 'page_count': 7, 'page_number': 1}
Text Preview:
 Abstract - A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between ma

In [59]:
# Section block helper functions

# build_chunk_page_metadata() is different from debug_utils.get_page_metadata():
def build_chunk_paper_metadata(paper):
    categories = paper.get("categories", [])

    return {
        "paper_id": paper.get("arxiv_id", "paper_0"),
        "title": paper.get("title", "untitled"),
        "primary_category": paper.get("primary_category", ""),
        "categories": ", ".join(categories) if isinstance(categories, list) else str(categories),
        "authors": ", ".join(paper.get("authors", [])) if paper.get("authors") else "",
        "published": paper.get("published", "")
    }


def build_chunk_page_metadata(page):
    metadata = page.get("metadata", {})

    return {
        "creation_date": metadata.get("creationDate", ""),
        "page_number": metadata.get("page_number", ""),
        "page_count": metadata.get("page_count", ""),
        "format": metadata.get("format", "")
    }


def remove_known_running_headers(text):
    if not text:
        return ""

    # Common journal/page header in the Vision Based Game Development PDF.
    text = re.sub(
        r"(?i)\(?IJCSIS\)?\s+International Journal of Computer Science and Information Security,\s*Vol\.\s*7,\s*No\.\s*1,\s*2010",
        " ",
        text
    )
    return text


def clean_block_text(text):
    return clean_text(remove_known_running_headers(text))


def should_keep_section(section_name, paper_title="", excluded_sections=None):
    excluded_sections = excluded_sections or set()
    normalized_section = section_name.strip()

    if normalized_section.title() in excluded_sections:
        return False

    return True


def build_section_block(section, text, paper, page, page_block_index):
    block_text = clean_block_text(text)
    if not block_text:
        return None

    return {
        "section": section,
        "text": block_text,
        "metadata": {
            **build_chunk_paper_metadata(paper),
            **build_chunk_page_metadata(page),
            "page_block_index": page_block_index
        }
    }

# wraps those parsed page sections with cleaned text and chunk metadata.
def extract_blocks_from_page(page, paper=None, excluded_sections=None):
    paper = paper or {}
    source_text = page.get("raw_text") or page.get("text", "")
    extracted_blocks = extract_section_blocks(source_text)
    blocks = []

    for block_index, block in enumerate(extracted_blocks):
        section = block.get("section", "").strip() or "Unknown"

        if not should_keep_section(section, paper.get("title", ""), excluded_sections):
            continue

        section_block = build_section_block(
            section=section,
            text=block.get("text", ""),
            paper=paper,
            page=page,
            page_block_index=len(blocks)
        )

        if section_block:
            blocks.append(section_block)

    return blocks


def append_continuation_to_previous_block(previous_block, page):
    continuation_text = clean_block_text(page.get("text") or page.get("raw_text", ""))

    if continuation_text:
        previous_block["text"] = f"{previous_block['text']} {continuation_text}".strip()

    return previous_block

# loops over pages from one paper and handles pages that continue a previous section.
def extract_blocks_from_paper(paper, excluded_sections=None):
    paper_blocks = []

    for page in paper.get("pages", []):
        page_blocks = extract_blocks_from_page(
            page,
            paper=paper,
            excluded_sections=excluded_sections
        )

        if page_blocks:
            paper_blocks.extend(page_blocks)
        elif paper_blocks:
            append_continuation_to_previous_block(paper_blocks[-1], page)
        else:
            fallback_block = build_section_block(
                section="Unknown",
                text=page.get("text") or page.get("raw_text", ""),
                paper=paper,
                page=page,
                page_block_index=0
            )
            if fallback_block:
                paper_blocks.append(fallback_block)

    return paper_blocks

#loops over many papers and combines their section blocks.
def extract_blocks_from_papers(cleaned_papers, excluded_sections=None):
    all_blocks = []

    for paper in cleaned_papers:
        all_blocks.extend(
            extract_blocks_from_paper(
                paper,
                excluded_sections=excluded_sections
            )
        )

    return all_blocks


In [60]:
# Section block configuration and execution
# Keep this cell small: change section_papers here when you want all papers vs one paper.

EXCLUDED_SECTIONS = {
    "References",
    "Acknowledgment",
    "Acknowledgements"
}

def select_papers_by_title(cleaned_papers, title_contains):
    title_contains = title_contains.lower()

    return [
        paper
        for paper in cleaned_papers
        if title_contains in paper.get("title", "").lower()
    ]

# Use all cleaned papers for the main pipeline.
# For single-paper debugging, uncomment the title filter below.
# SECTION_PAPER_TITLE = "Vision Based Game Development"
# section_papers = select_papers_by_title(cleaned_papers, SECTION_PAPER_TITLE)
# if not section_papers:
#     raise ValueError(f"No cleaned paper title contains: {SECTION_PAPER_TITLE}")

section_papers = cleaned_papers
all_blocks = extract_blocks_from_papers(section_papers, excluded_sections=EXCLUDED_SECTIONS)


In [61]:
preview_section_blocks(all_blocks)



PAGE: 1
SECTION: Vision Based Game Development Using Human Computer Interaction
TEXT SAMPLE: Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India

PAGE: 1
SECTION: I. INTRODUCTION
TEXT SAMPLE: One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video camera and translates them into the movements of the mouse pointer on the screen. The tip of the user's nose can be tracked and captured with a webcam and monitor its movements in order to translate it 

PAGE: 2
SECTION: II. RELATED WORK
TEXT SAMPLE: With the growth of attention about computer vision, the interest in HCI has increased proportionally. Different human features and monitoring devices were used to achieve HCI, but during our resear

In [62]:
# chunking the cleaned text for spacy using langchain text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter


def build_text_splitter(chunk_size=500, chunk_overlap=200):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,  # may need to reduce when scaling up to avoid too much redundancy
        separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""]
    )


def chunk_section_blocks(blocks, text_splitter=None, starting_chunk_index=0):
    text_splitter = text_splitter or build_text_splitter()
    chunks = []
    global_chunk_index = starting_chunk_index

    for block in blocks:
        block_text = block.get("text", "")
        if not block_text.strip():
            continue

        splits = text_splitter.split_text(block_text)

        for page_chunk_index, split_text in enumerate(splits):
            split_text = re.sub(r"^[\s\.,;:-]+", "", split_text).strip()

            if split_text:
                metadata = {
                    **block.get("metadata", {}),
                    "section": block.get("section", "Unknown"),
                    "page_chunk_index": page_chunk_index,
                    "chunk_index": global_chunk_index
                }

                chunks.append({
                    "text": split_text,
                    "metadata": metadata
                })
                global_chunk_index += 1

    return chunks


def chunk_cleaned_papers(cleaned_papers, text_splitter=None, excluded_sections=None):
    blocks = extract_blocks_from_papers(
        cleaned_papers,
        excluded_sections=excluded_sections
    )
    return chunk_section_blocks(blocks, text_splitter=text_splitter)


# text_splitter = build_text_splitter() remove
text_splitter = build_text_splitter(
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap
)
chunks = chunk_section_blocks(all_blocks, text_splitter=text_splitter)

print("Total chunks:", len(chunks))
print(chunks[0] if chunks else "No chunks created")


Total chunks: 517
{'text': 'Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India', 'metadata': {'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'page_block_index': 0, 'section': 'Vision Based Game Development Using Human Computer Interaction', 'page_chunk_index': 0, 'chunk_index': 0}}


In [63]:
# separating metadata from text for NER to avoid noise
def text_for_ner(chunk):
    text = (chunk.get("text") or "")
    md = chunk.get("metadata", {})

    # Clean special chars that confuse NER
    text = re.sub(r"[*_`~^—–]+", " ", text)

    # Remove leading punctuation/whitespace
    text = re.sub(r"^[\s\.,;:-]+", "", text)

    if md.get("page_number") == 1:
        title = (md.get("title") or "")
        if title:
            text = re.sub(re.escape(title), " ", text, flags=re.IGNORECASE)

        authors = md.get("authors", "")
        if isinstance(authors, str):
            author_list = [a.strip() for a in authors.split(",") if a.strip()]
        elif isinstance(authors, list):
            author_list = [str(a).strip() for a in authors if str(a).strip()]
        else:
            author_list = []

        for author in author_list:
            text = re.sub(rf"\b{re.escape(author)}\b", " ", text, flags=re.IGNORECASE)

        # Keep only text after Abstract if present
        parts = re.split(r"\babstract\b[:\-\s]*", text, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) == 2:
            text = parts[1]

    return re.sub(r"\s+", " ", text).strip()

#### Entity Extraction

In [64]:
#using spacy model for NER

import spacy

nlp = spacy.load("en_core_web_md") # using at the moment since it has better NER performance than the small mode
# nlp = spacy.load("en_core_web_sm") # would eventually switch when increase research papers for faster performance

In [65]:
def clean_entities(raw_entities):
    cleaned = []
    seen = set()

    for ent_text, ent_label in raw_entities:
        ent_text = ent_text.strip()

        # skip empty entities
        if not ent_text:
            continue

        # remove very short fragments like "I."
        if len(ent_text) < 3:
            continue

        # remove pure numbers
        if re.fullmatch(r"\d+(\.\d+)?", ent_text):
            continue

        # skip ordinal/cardinal noise
        if ent_label in {"CARDINAL", "ORDINAL"}:
            continue

        # remove mostly punctuation fragments
        if re.fullmatch(r"[^\w]+", ent_text):
            continue

        key = (ent_text.lower(), ent_label)
        if key not in seen:
            seen.add(key)
            cleaned.append((ent_text, ent_label))

    return cleaned

In [66]:
# # updating metadata with NER results
def enrich_chunks_with_entities(chunks, nlp_model=None, batch_size=32):
    nlp_model = nlp_model or nlp

    ner_texts = [text_for_ner(chunk) for chunk in chunks]
    docs = nlp_model.pipe(ner_texts, batch_size=batch_size)

    for chunk, doc, ner_text in zip(chunks, docs, ner_texts):
        raw_entities = [(ent.text, ent.label_) for ent in doc.ents]
        cleaned = clean_entities(raw_entities)

        chunk["metadata"]["entities"] = ", ".join(text for text, _ in cleaned)
        chunk["metadata"]["entity_labels"] = ", ".join(label for _, label in cleaned)
        chunk["metadata"]["ner_text"] = ner_text

    return chunks

chunks = enrich_chunks_with_entities(chunks, nlp_model=nlp, batch_size=32)

In [67]:
# printing the metadata of the first chunk to verify NER results are stored
print(chunks[0]["metadata"])

{'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'page_block_index': 0, 'section': 'Vision Based Game Development Using Human Computer Interaction', 'page_chunk_index': 0, 'chunk_index': 0, 'entities': 'Bharath University St.Joseph College of Engineering Bharath University Chennai, India Chennai, India', 'entity_labels': 'ORG, GPE, GPE', 'ner_text': 'Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India'}


In [68]:
METADATA_KEYS = [
    "paper_id", "title", "section", "primary_category", "categories",
    "authors", "published", "creation_date", "page_number", "page_count",
    "format", "page_block_index", "page_chunk_index", "chunk_index",
    "entities", "entity_labels"
]

def build_final_chunks(chunks):
    final_chunks = []

    for i, chunk in enumerate(chunks):
        metadata = {
            key: chunk["metadata"].get(key, "")
            for key in METADATA_KEYS
        }

        if not metadata["paper_id"]:
            metadata["paper_id"] = "paper_0"

        if metadata["chunk_index"] == "":
            metadata["chunk_index"] = i

        final_chunks.append({
            "text": chunk["text"],
            "metadata": metadata,
        })

    return final_chunks


final_chunks = build_final_chunks(chunks)

for i, chunk in enumerate(final_chunks[:5]):
    print("\nFinal Chunk", i, "metadata:", chunk["metadata"])
    print("Final Chunk", i, "text preview:", chunk["text"][:180])


Final Chunk 0 metadata: {'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'Vision Based Game Development Using Human Computer Interaction', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'page_block_index': 0, 'page_chunk_index': 0, 'chunk_index': 0, 'entities': 'Bharath University St.Joseph College of Engineering Bharath University Chennai, India Chennai, India', 'entity_labels': 'ORG, GPE, GPE'}
Final Chunk 0 text preview: Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India

Final Chunk 1 metadata: {'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 

#### EMBEDDING WITH OPENAI AND STORING IN CHROMADB

In [69]:
from openai import OpenAI
import os
import dotenv
import chromadb

dotenv.load_dotenv()  # Load environment variables from .env file

client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
embedding_model = "text-embedding-3-small" # will change to text-embedding-3-large when increase research papers for better performance

# using persistent chroma db to store the embeddings and metadata for later retrieval and analysis
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection_name = "research_papers"

try:
    chroma_client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")
except:
    print(f"No existing collection named: {collection_name}")

collection = chroma_client.get_or_create_collection(
    name=collection_name,
    configuration={"hnsw": {"space": "cosine"}}
)
print("Chroma collection ready.")

No existing collection named: research_papers
Chroma collection ready.


In [92]:
#preparing data for insertion into chroma db
def embed_chunks(final_chunks, config):
    documents = [chunk["text"] for chunk in final_chunks]
    metadatas = [chunk["metadata"] for chunk in final_chunks]
    ids = [
        f"{chunk['metadata']['paper_id']}_chunk_{chunk['metadata']['chunk_index']}"
        for chunk in final_chunks
    ]

    all_embeddings = []

    for i in range(0, len(documents), config.batch_size):
        batch_docs = documents[i:i + config.batch_size]

        batch_embeddings = get_embeddings(
            batch_docs,
            model=config.embedding_model
        )

        all_embeddings.extend(batch_embeddings)
        print(f"Embedded chunks {i} to {i + len(batch_docs) - 1}")

    # stats
    total_embeddings = len(all_embeddings)
    embedding_dim = len(all_embeddings[0]) if all_embeddings else 0

    print("\n--- Embedding Stats ---")
    print("Total embeddings created:", total_embeddings)
    print("Embedding dimension:", embedding_dim)

    return documents, metadatas, ids, all_embeddings

#storing embeddings and metadata in chroma db
def store_chunks_in_chroma(collection, documents, metadatas, ids, embeddings):
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )

    print(f"Stored {len(documents)} chunks in ChromaDB.")

    return len(documents)

In [77]:
documents, metadatas, ids, embeddings = embed_chunks(final_chunks, config)

stored_count = store_chunks_in_chroma(
    collection,
    documents,
    metadatas,
    ids,
    embeddings
)

print("\nStored count:", stored_count)

Embedded chunks 0 to 19
Embedded chunks 20 to 39
Embedded chunks 40 to 59
Embedded chunks 60 to 79
Embedded chunks 80 to 99
Embedded chunks 100 to 119
Embedded chunks 120 to 139
Embedded chunks 140 to 159
Embedded chunks 160 to 179
Embedded chunks 180 to 199
Embedded chunks 200 to 219
Embedded chunks 220 to 239
Embedded chunks 240 to 259
Embedded chunks 260 to 279
Embedded chunks 280 to 299
Embedded chunks 300 to 319
Embedded chunks 320 to 339
Embedded chunks 340 to 359
Embedded chunks 360 to 379
Embedded chunks 380 to 399
Embedded chunks 400 to 419
Embedded chunks 420 to 439
Embedded chunks 440 to 459
Embedded chunks 460 to 479
Embedded chunks 480 to 499
Embedded chunks 500 to 516

--- Embedding Stats ---
Total embeddings created: 517
Embedding dimension: 1536
Stored 517 chunks in ChromaDB.

Stored count: 517


### Testing Retrieval

In [78]:
# def retrieve_chunks(query, n_results=3):
#     query_embedding = get_embeddings([query], model=embedding_model)[0]
#     results = collection.query(
#         query_embeddings=[query_embedding],
#         n_results=n_results
#     )
#     return results
def retrieve_chunks(query, n_results=3, filter_category=None):
    query_embedding = get_embeddings(
        [query],
        model=config.embedding_model
    )[0]

    where = None
    if filter_category:
        where = {"primary_category": filter_category}

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where
    )

    return results

In [79]:
test_queries = [
    "What is the main purpose of the proposed HCI system?",
    "What is Hough transform and how is it used in this paper?",
    "How does blink detection work in the eye ROI?"
]

for query in test_queries:
    print("\n" + "="*50)
    print(f"\nSearch results for query: '{query}'")
    results = retrieve_chunks(query, n_results=3)

    for i in range(len(results["documents"][0])):
        print(f"\nResult {i+1}")
        print("ID:", results["ids"][0][i])
        print("Metadata:", results["metadatas"][0][i])
        print("Text:", results["documents"][0][i][:400])
    print("\n" + "="*50)



Search results for query: 'What is the main purpose of the proposed HCI system?'

Result 1
ID: 1002.2191v1_chunk_1
Metadata: {'primary_category': 'cs.HC', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'categories': 'cs.HC, cs.CV, cs.MM', 'published': '2010-02-10T19:46:07Z', 'entities': 'HCI', 'paper_id': '1002.2191v1', 'page_chunk_index': 0, 'format': 'PDF 1.4', 'page_block_index': 1, 'creation_date': "D:20100202231332+05'00'", 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'page_count': 7, 'page_number': 1, 'entity_labels': 'ORG', 'chunk_index': 1}
Text: One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video camera and translates them into the movements of the mouse pointer 

### SEMANTIC SEARCH ENGINE


In [80]:
#entity normalization function to create a consistent set of entities for analysis and retrieval
def normalize_entity_set(entities: str):
    return {
        e.strip().lower()
        for e in entities.split(",")
        if e.strip()
    }

In [81]:
# entity overlap function to compute the number of shared entities between a chunk and a query for relevance scoring and ranking
def compute_entity_overlap(entities: str, query: str):
    entity_set = normalize_entity_set(entities)
    query_terms = {term.lower().strip() for term in query.split() if term.strip()}
    return len(entity_set.intersection(query_terms))

In [82]:
# hybrid scoring to combine cosine similarity with entity overlap for improved relevance ranking of search results
def compute_hybrid_score(cosine_similarity, entity_overlap, use_hybrid=False, entity_bonus=0.5):
    if cosine_similarity is None:
        return None
    if not use_hybrid:
        return cosine_similarity
    return cosine_similarity + (entity_overlap * entity_bonus)

In [83]:
def semantic_search(query, n_results=5, min_similarity=0.25, filter_category=None, use_hybrid=False, entity_bonus=0.5):
    
    # results = retrieve_chunks(query, n_results=n_results)
    results = retrieve_chunks(
    query,
    n_results=n_results,
    filter_category=filter_category
)
    formatted_results = []

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    ids = results.get("ids", [[]])[0]
    distances = results.get("distances", [[]])[0] if "distances" in results else []

    # this filter is for experimentation to improve llm response accuracy (the idea is if the user specify a category, filter the chunks to only that category to reduce noise for the llm)
    filter_category_norm = filter_category.lower().strip() if filter_category else None

    for i, doc_text in enumerate(documents):
        metadata = metadatas[i] if i < len(metadatas) else {}
        chunk_id = ids[i] if i < len(ids) else f"chunk_{i}"

        cosine_distance = distances[i] if i < len(distances) else None
        cosine_similarity = 1 - cosine_distance if cosine_distance is not None else None

        if cosine_similarity is not None and cosine_similarity < min_similarity:
            continue

        primary_category = metadata.get("primary_category", "")
        # if filter_category_norm and primary_category.lower().strip() != filter_category_norm:
        #     continue

        entities = metadata.get("entities", "")
        print(f"\nEntities for chunk {chunk_id}: {entities}")

        entity_overlap = compute_entity_overlap(entities, query)
        hybrid_score = compute_hybrid_score(
            cosine_similarity,
            entity_overlap,
            use_hybrid=use_hybrid,
            entity_bonus=entity_bonus
        )

        formatted_results.append({
            "rank": i + 1,
            "chunk_id": chunk_id,
            "cosine_distance": cosine_distance,
            "cosine_similarity": cosine_similarity,
            "hybrid_score": hybrid_score,
            "entity_overlap": entity_overlap,
            "text": doc_text,
            "paper_id": metadata.get("paper_id", ""),
            "title": metadata.get("title", ""),
            "section": metadata.get("section", ""),
            "primary_category": primary_category,
            "categories": metadata.get("categories", ""),
            "authors": metadata.get("authors", ""),
            "published": metadata.get("published", ""),
            "page_number": metadata.get("page_number", ""),
            "page_block_index": metadata.get("page_block_index", ""),
            "page_chunk_index": metadata.get("page_chunk_index", ""),
            "chunk_index": metadata.get("chunk_index", ""),
            "entities": entities,
            "entity_labels": metadata.get("entity_labels", "")
        })

    if use_hybrid:
        formatted_results = sorted(
            formatted_results,
            key=lambda x: x["hybrid_score"] if x["hybrid_score"] is not None else -1,
            reverse=True
        )
        for idx, result in enumerate(formatted_results, start=1):
            result["rank"] = idx

    return formatted_results


In [84]:
from debug_utils import display_search_results


In [85]:
def semantic_search_engine(query, n_results=5):
    results = semantic_search(query, n_results=n_results)
    return {
        "query": query,
        "total_results": len(results),
        "results": results
    }

In [86]:
search_output = semantic_search_engine("What is the main purpose of the proposed HCI system?", n_results=3)

print("Query:", search_output["query"])
print("Total results:", search_output["total_results"])
display_search_results(search_output["results"])


Entities for chunk 1002.2191v1_chunk_1: HCI

Entities for chunk 1002.2191v1_chunk_8: HCI

Entities for chunk 1002.2191v1_chunk_13: HCI, EOG
Query: What is the main purpose of the proposed HCI system?
Total results: 3
Rank: 1
Paper ID: 1002.2191v1
Title: Vision Based Game Development Using Human Computer Interaction
Section: I. INTRODUCTION
Primary Category: cs.HC
Categories: cs.HC, cs.CV, cs.MM
Authors: S. Sumathi, S. K. Srivatsa, M. Uma Maheswari
Published: 2010-02-10T19:46:07Z
Page: 1 | Block: 1 | Page Chunk: 0 | Global Chunk: 1
Cosine Similarity (approx): 0.6012195944786072
Entities: HCI
Entity Labels: ORG
Preview: One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video ca

Rank: 2
Paper ID: 1002.2191v1
Title: Vision Based Game Development Usin

### RAG PIPELINE IMPLEMENTATION

In [87]:
RAG_SYSTEM_PROMPT = """
You are an academic research assistant for an enterprise knowledge mining system built on arXiv papers.

Instructions:
- Answer questions only from the retrieved context.
- Treat the retrieved text as excerpts from research papers.
- Prefer precise academic wording.
- Summarize contributions, methods, datasets, results, or limitations only if they are supported by the context.
- If the answer is incomplete or missing from the context, say that explicitly.
- Do not include source tags or citation brackets such as [Source 1] in the answer text. 
- Write only the answer and ensure it is formatted concisely and clearly.
"""
# not including source tags so the answer is cleaner, the sources will be provided separately in the metadata for each retrieved chunk

In [88]:
# function to generate answer from retrieved context using the RAG system prompt
def generate_answer(query, context, max_tokens=200, temperature=0.2):
    response = client.chat.completions.create(
        model= config.rag_model,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Use the retrieved research-paper context below to answer the question.\n\n"
                    f"Context:\n{context}\n\n"
                    f"Question: {query}\n\n"
                    "Answer:"
                )
            }
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )

    return response.choices[0].message.content.strip()

In [89]:
# function to perform RAG query and return answer with sources
def rag_query(query, n_results=3, min_similarity=0.25, filter_category=None, use_hybrid=False, entity_bonus=0.5):
    
    search_results = semantic_search(query=query, n_results=n_results, min_similarity=min_similarity, filter_category=filter_category, use_hybrid=use_hybrid, entity_bonus=entity_bonus)

    if not search_results:
        return {
            "query": query,
            "answer": "I cannot answer this question based on the provided context. No relevant context was found.",
            "sources": []
        }

    context_parts = []
    sources = []

    for i, result in enumerate(search_results):
        context_parts.append(
            f"[Source {i+1}: {result['title']}, "
            f"Section: {result['section']}, "
            f"Page {result['page_number']}, "
            f"Chunk {result['chunk_index']}]\n"
            f"{result['text']}"
        )

        sources.append({
            "source_number": i + 1,
            "chunk_id": result["chunk_id"],
            "paper_id": result["paper_id"],
            "title": result["title"],
            "section": result["section"],
            "primary_category": result["primary_category"],
            "categories": result["categories"],
            "page_number": result["page_number"],
            "chunk_index": result["chunk_index"],
            "cosine_similarity": result["cosine_similarity"],
            "hybrid_score": result["hybrid_score"],
            "entity_overlap": result["entity_overlap"]
        })

    context = "\n\n".join(context_parts)
    answer = generate_answer(query, context)

    return {
        "query": query,
        "answer": answer,
        "sources": sources
    }

In [90]:
queries=[
    # # testing how it answers questions that are directly related to the paper
    # "What is the main purpose of the proposed HCI system?", 
    # # testing how it answer partially related questions
    # "What hardware and software requirements are needed to run the proposed HCI system?", 
    # # testing how it answers unrelated questions
    # "Who is the Last Airbender?",   
    # #entity based questions to test the entity normalization and filtering
    # "What role does the nose tip play in the interface?",
    # "How does blink detection work in the system?",
    # "What does the paper say about USB cameras?"
    "What is HCI and how is it used in the system?"
]

for q in queries:
    print("\n" + "="*50)
    print(f"\nRAG response for query: '{q}'")
    response = rag_query(q, n_results=3)
    print("Answer:\n", response["answer"])
    print("\nSources:")
    for s in response["sources"]:
        print(s)



RAG response for query: 'What is HCI and how is it used in the system?'

Entities for chunk 1002.2191v1_chunk_1: HCI

Entities for chunk 1002.2191v1_chunk_8: HCI

Entities for chunk 1002.2191v1_chunk_9: 1, 2010, HCI
Answer:
 Human-Computer Interface (HCI) is the point of communication between a human and a computer, aiming to use human features to interact with the computer. In the system, HCI tracks the user's movements with a video camera and translates these movements into corresponding movements of the mouse pointer on the screen.

Sources:
{'source_number': 1, 'chunk_id': '1002.2191v1_chunk_1', 'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'page_number': 1, 'chunk_index': 1, 'cosine_similarity': 0.6645033955574036, 'hybrid_score': 0.6645033955574036, 'entity_overlap': 1}
{'source_number': 2, 'chunk_id': '1002.2191v1_chunk_8', 'pa

### EXPERIMENTAL : RERANKING CHUNKS BASED ON ENTITY OVERLAP

In [91]:
entity_queries = [
    ("What is HCI and how is it used in the system?", {"use_hybrid": True}),
    # ("What role does the nose tip play in the interface?", {"use_hybrid": True}),
    # ("How does blink detection work in the system?", {"use_hybrid": True}),
    # ("What does the paper say about USB cameras?", {"use_hybrid": True}),
]

for q, opts in entity_queries:
    print("\n" + "=" * 50)
    print(f"\nRAG response for query: '{q}'")
    response = rag_query(q, n_results=3, **opts)
    print("Answer:\n", response["answer"])
    print("\nSources:")
    for s in response["sources"]:
        print(s)



RAG response for query: 'What is HCI and how is it used in the system?'

Entities for chunk 1002.2191v1_chunk_1: HCI

Entities for chunk 1002.2191v1_chunk_8: HCI

Entities for chunk 1002.2191v1_chunk_9: 1, 2010, HCI
Answer:
 Human-Computer Interface (HCI) is the point of communication between a human and a computer, aiming to use human features to interact with the computer. In the system, HCI tracks the user's movements using a video camera and translates these movements into corresponding movements of the mouse pointer on the screen.

Sources:
{'source_number': 1, 'chunk_id': '1002.2191v1_chunk_1', 'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'page_number': 1, 'chunk_index': 1, 'cosine_similarity': 0.6645375490188599, 'hybrid_score': 1.1645375490188599, 'entity_overlap': 1}
{'source_number': 2, 'chunk_id': '1002.2191v1_chunk_8', 'p

This test is to explore whether NER-derived metadata could provide additional retrieval benefit (other than enriching metadata) through hybrid re-ranking. The use_hybrid=True to re rank chunks that have higher entity overlap after retrieving semantically as usual. However, there is currently no significant improvement in re-ranking the chunks based on entity overlap. The general semantic search flow using `retrieve_chunks` already performs good without this enhancement.

### EXPERIMENTAL : FILTER CATEGORY


This will be tested when we scale up documents once the categories vary among documents.